In [1]:
import sys
sys.path.append("..")

import joblib
import pandas as pd

from src.features import FEATURE_COLUMNS

In [2]:
scaler = joblib.load("../models/scaler.joblib")
kmeans_model = joblib.load("../models/kmeans_model.joblib")
persona_labels = joblib.load("../models/persona_labels.joblib")

print("Feature columns (order matters):", FEATURE_COLUMNS)
print("KMeans clusters:", kmeans_model.n_clusters)
print("Personas:", persona_labels)

Feature columns (order matters): ['Recency', 'Frequency', 'Monetary', 'AvgBasketValue', 'AvgItemsPerBasket', 'UniqueProducts', 'CancellationRate', 'CustomerLifespanDays']
KMeans clusters: 4
Personas: {0: 'Champions (Recent, Frequent, High-Spend)', 1: 'Regular Customers', 2: 'At-Risk / Churned Customers', 3: 'High Cancellation Rate (Risk Watch)'}


In [3]:
# IMPORTANT: the scaler was fit on LOG-TRANSFORMED values for these columns
# (see notebook 03) — any new customer's raw values must be log1p'd on these
# same columns before scaling, or predictions will be wrong. We record the
# list in the bundle itself so the FastAPI/Streamlit serving code doesn't
# have to guess or duplicate this list by hand.
log_columns = ["Frequency", "Monetary", "AvgBasketValue", "AvgItemsPerBasket", "UniqueProducts"]

pipeline_bundle = {
    "scaler": scaler,
    "kmeans_model": kmeans_model,
    "persona_labels": persona_labels,
    "feature_columns": FEATURE_COLUMNS,
    "log_columns": log_columns,
}

joblib.dump(pipeline_bundle, "../models/clustering_pipeline.joblib")
print("Saved models/clustering_pipeline.joblib")

Saved models/clustering_pipeline.joblib


In [4]:
import numpy as np

bundle = joblib.load("../models/clustering_pipeline.joblib")

new_customer = pd.DataFrame([{
    "Recency": 12,
    "Frequency": 8,
    "Monetary": 650.0,
    "AvgBasketValue": 81.25,
    "AvgItemsPerBasket": 6.5,
    "UniqueProducts": 22,
    "CancellationRate": 0.05,
    "CustomerLifespanDays": 210,
}])[bundle["feature_columns"]]

# Apply the same log1p transform used at training time before scaling
new_customer_transformed = new_customer.copy()
for col in bundle["log_columns"]:
    new_customer_transformed[col] = np.log1p(new_customer_transformed[col])

scaled = bundle["scaler"].transform(new_customer_transformed)
cluster_id = int(bundle["kmeans_model"].predict(scaled)[0])
persona = bundle["persona_labels"][cluster_id]

print(f"Predicted cluster: {cluster_id}")
print(f"Persona: {persona}")

Predicted cluster: 1
Persona: Regular Customers
